In [ ]:
!pip install requests pandas

In [ ]:
from google.colab import userdata
APP_ID = userdata.get('ADZUNA_APP_ID')
APP_KEY = userdata.get('ADZUNA_APP_KEY')

In [ ]:
import requests
import pandas as pd
import time

COUNTRY = "in"  # India
SEARCH_TERMS = ["data analyst", "data scientist", "business intelligence analyst"]
RESULTS_PER_PAGE = 50
PAGES_PER_TERM = 4  # 4 pages x 50 = up to 200 postings per search term

BASE_URL = f"https://api.adzuna.com/v1/api/jobs/{COUNTRY}/search"

all_jobs = []

for term in SEARCH_TERMS:
    print(f"Pulling postings for: {term}")
    for page in range(1, PAGES_PER_TERM + 1):
        params = {
            "app_id": APP_ID,
            "app_key": APP_KEY,
            "what": term,
            "results_per_page": RESULTS_PER_PAGE,
            "content-type": "application/json",
        }
        url = f"{BASE_URL}/{page}"
        resp = requests.get(url, params=params, timeout=30)

        if resp.status_code != 200:
            print(f"  Page {page}: request failed ({resp.status_code}) — {resp.text[:200]}")
            break

        data = resp.json()
        results = data.get("results", [])
        if not results:
            print(f"  Page {page}: no more results, stopping this term.")
            break

        for job in results:
            all_jobs.append({
                "search_term": term,
                "title": job.get("title"),
                "company_name": (job.get("company") or {}).get("display_name"),
                "location": (job.get("location") or {}).get("display_name"),
                "salary_min": job.get("salary_min"),
                "salary_max": job.get("salary_max"),
                "contract_type": job.get("contract_type"),
                "contract_time": job.get("contract_time"),
                "created": job.get("created"),
                "category": (job.get("category") or {}).get("label"),
                "description": job.get("description"),
                "redirect_url": job.get("redirect_url"),
            })

        print(f"  Page {page}: {len(results)} postings pulled (running total: {len(all_jobs)})")
        time.sleep(1)  # be polite to the API / stay within rate limits

df = pd.DataFrame(all_jobs)
df.drop_duplicates(subset=["title", "company_name", "location", "created"], inplace=True)

output_path = "adzuna_india_data_analytics_jobs.csv"
df.to_csv(output_path, index=False)
print(f"\nDone. Saved {len(df)} unique postings to {output_path}")

Pulling postings for: data analyst
  Page 1: 50 postings pulled (running total: 50)
  Page 2: 50 postings pulled (running total: 100)
  Page 3: 50 postings pulled (running total: 150)
  Page 4: 50 postings pulled (running total: 200)
Pulling postings for: data scientist
  Page 1: 50 postings pulled (running total: 250)
  Page 2: 50 postings pulled (running total: 300)
  Page 3: 50 postings pulled (running total: 350)
  Page 4: 50 postings pulled (running total: 400)
Pulling postings for: business intelligence analyst
  Page 1: 50 postings pulled (running total: 450)
  Page 2: 50 postings pulled (running total: 500)
  Page 3: 50 postings pulled (running total: 550)
  Page 4: 13 postings pulled (running total: 563)

Done. Saved 553 unique postings to adzuna_india_data_analytics_jobs.csv


In [ ]:
import re
import pandas as pd

INDIA_INPUT_PATH = "adzuna_india_data_analytics_jobs.csv"
US_INPUT_PATH = "data_analyst_jobs_cleaned.xlsx"

INDIA_OUTPUT_PATH = "india_jobs_cleaned.csv"
COMPARISON_OUTPUT_PATH = "us_vs_india_skill_comparison.csv"
INDIA_LOCATIONS_OUTPUT_PATH = "india_top_locations.csv"
INDIA_SALARY_OUTPUT_PATH = "india_salary_by_role.csv"

india = pd.read_csv(INDIA_INPUT_PATH)
us = pd.read_excel(US_INPUT_PATH)

before = len(india)
india = india.drop_duplicates(subset=['title', 'company_name', 'location', 'description'])
print(f"Deduplication: {before} -> {len(india)} rows ({before - len(india)} removed)")

def split_location(loc):
    if pd.isna(loc) or loc.strip() == 'India':
        return pd.Series(['Unspecified', None])
    parts = [p.strip() for p in loc.split(',')]
    city = parts[0]
    state = parts[1] if len(parts) > 1 else None
    return pd.Series([city, state])

india[['location_city', 'location_state']] = india['location'].apply(split_location)
india['is_remote'] = india['location_city'].apply(
    lambda x: 'Unspecified/Remote' if x == 'Unspecified' else 'On-site/Hybrid'
)

india['salary_avg'] = india[['salary_min', 'salary_max']].mean(axis=1)
india['has_salary'] = india['salary_avg'].notna().map({True: 'Yes', False: 'No'})


skills_us_raw = us['skills_clean'].dropna().str.split(';').explode().str.strip().str.lower()
ambiguous = ['c', 'go', 'r']
skill_vocab = sorted(set(s for s in skills_us_raw if s and s not in ambiguous))

def extract_skills(text):
    if pd.isna(text):
        return ''
    text_l = text.lower()
    found = []
    for skill in skill_vocab:
        pattern = r'\b' + re.escape(skill) + r'\b'
        if re.search(pattern, text_l):
            found.append(skill)
    for skill in ambiguous:
        pattern = r'\b' + re.escape(skill) + r'\b'
        if re.search(pattern, text_l):
            found.append(skill)
    return ';'.join(sorted(set(found)))

india['skills_clean'] = india['description'].apply(extract_skills)
india['has_skills'] = india['skills_clean'].apply(lambda x: 'Yes' if x else 'No')

pct_with_skills = (india['has_skills'] == 'Yes').mean() * 100
print(f"India postings with at least 1 extracted skill: {pct_with_skills:.1f}%")

india_out = india[[
    'search_term', 'title', 'company_name', 'location_city', 'location_state',
    'is_remote', 'category', 'contract_time', 'salary_min', 'salary_max',
    'salary_avg', 'has_salary', 'skills_clean', 'has_skills', 'created', 'redirect_url'
]].copy()
india_out.insert(0, 'posting_id', range(1, len(india_out) + 1))
india_out.to_csv(INDIA_OUTPUT_PATH, index=False)
print(f"Saved {INDIA_OUTPUT_PATH} ({len(india_out)} rows)")


india_skills_long = india_out[['posting_id', 'skills_clean']].copy()
india_skills_long = india_skills_long.assign(
    skill_name=india_skills_long['skills_clean'].fillna('').str.split(';')
).explode('skill_name')
india_skills_long = india_skills_long[india_skills_long['skill_name'].fillna('') != '']
india_skill_counts = india_skills_long.groupby('skill_name').size().reset_index(name='posting_count')
india_skill_counts['pct_of_postings'] = round(100 * india_skill_counts['posting_count'] / len(india_out), 1)
india_skill_counts = india_skill_counts.sort_values('posting_count', ascending=False)

us_skills_long = us[['skills_clean']].copy()
us_skills_long = us_skills_long.assign(
    skill_name=us_skills_long['skills_clean'].fillna('').str.split(';')
).explode('skill_name')
us_skills_long = us_skills_long[us_skills_long['skill_name'].fillna('') != '']
us_skill_counts = us_skills_long.groupby('skill_name').size().reset_index(name='posting_count')
us_skill_counts['pct_of_postings'] = round(100 * us_skill_counts['posting_count'] / len(us), 1)
us_skill_counts = us_skill_counts.sort_values('posting_count', ascending=False)

comparison = pd.merge(
    us_skill_counts[['skill_name', 'pct_of_postings']].rename(columns={'pct_of_postings': 'us_pct'}),
    india_skill_counts[['skill_name', 'pct_of_postings']].rename(columns={'pct_of_postings': 'india_pct'}),
    on='skill_name', how='outer'
).fillna(0)
comparison = comparison.sort_values('us_pct', ascending=False)
comparison.to_csv(COMPARISON_OUTPUT_PATH, index=False)
print(f"Saved {COMPARISON_OUTPUT_PATH} ({len(comparison)} rows)")

india_out.groupby('location_city').size().reset_index(name='posting_count') \
    .sort_values('posting_count', ascending=False).head(15) \
    .to_csv(INDIA_LOCATIONS_OUTPUT_PATH, index=False)
print(f"Saved {INDIA_LOCATIONS_OUTPUT_PATH}")

india_salary = india_out[india_out['has_salary'] == 'Yes'].groupby('search_term')['salary_avg'].agg(['count', 'mean']).reset_index()
india_salary.columns = ['search_term', 'posting_count', 'avg_salary_inr']
india_salary['avg_salary_inr'] = india_salary['avg_salary_inr'].round(0)
india_salary.to_csv(INDIA_SALARY_OUTPUT_PATH, index=False)
print(f"Saved {INDIA_SALARY_OUTPUT_PATH}")

print("\nTop 10 India skills:")
print(india_skill_counts.head(10).to_string(index=False))
print("\nUS vs India skill comparison (top 15 by US demand):")
print(comparison.head(15).to_string(index=False))
print("\nIndia salary by role:")
print(india_salary.to_string(index=False))

Deduplication: 553 -> 544 rows (9 removed)
India postings with at least 1 extracted skill: 25.0%
Saved india_jobs_cleaned.csv (544 rows)
Saved us_vs_india_skill_comparison.csv (134 rows)
Saved india_top_locations.csv
Saved india_salary_by_role.csv

Top 10 India skills:
skill_name  posting_count  pct_of_postings
       sql             69             12.7
    python             60             11.0
     excel             34              6.2
         r             23              4.2
   tableau             20              3.7
    hadoop              9              1.7
     spark              9              1.7
tensorflow              9              1.7
    pandas              8              1.5
     numpy              7              1.3

US vs India skill comparison (top 15 by US demand):
skill_name  us_pct  india_pct
       sql    49.4       12.7
     excel    31.8        6.2
    python    30.3       11.0
  power_bi    27.5        0.0
   tableau    26.5        3.7
         r    18.1      